# CHiRPE Tutorial: Fused String-Input ONNX With Transcript Voting

This notebook shows how to use the fixed-slot fused CHiRPE ONNX model where the tokenizer, classifier, and transcript-level voting all run inside ONNX.

Pipeline:

```text
segment summaries -> tokenizer inside ONNX -> classifier inside ONNX -> transcript voting inside ONNX
```


## Input Contract

One ONNX request represents one transcript. The model expects exactly 15 segment slots:

```text
text: tensor(string)[15]
segment_mask: tensor(float)[15]
```

If a transcript has fewer than 15 segments, pad the remaining slots with empty strings and mask them out with `0.0`.

Example for 2 real segments:

```python
text = ["segment 1 summary", "segment 2 summary"] + [""] * 13
segment_mask = [1.0, 1.0] + [0.0] * 13
```

The padded slots still flow through the graph, but they are ignored by transcript-level voting because their mask value is `0.0`.


## What Is Inside ONNX?

Inside ONNX:

- BERT tokenizer custom op from ONNX Runtime Extensions
- Transformer classifier
- Segment-level softmax and argmax
- Masked average transcript voting
- Masked majority transcript voting

Outside ONNX:

- Raw transcript loading
- PSYCHS segmentation
- Optional summarization
- Packing summaries into `text[15]` and `segment_mask[15]`


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

In [ ]:
import onnxruntime as ort
from onnxruntime_extensions import get_library_path

from chirpe.data.preprocessor import TranscriptPreprocessor

## Choose A Fused Voting Model

The examples below default to BERT. Change `BACKBONE` to `clinicalbert` or `mentalbert` if those fixed-slot models have been generated.


In [ ]:
BACKBONE = "bert"  # bert | clinicalbert | mentalbert
MAX_SEGMENTS = 15

MODEL_NAME = f"chirpe_{BACKBONE}_string_voting"
MODEL_DIR = REPO_ROOT / "outputs/string_onnx_fused_voting" / MODEL_NAME
MODEL_PATH = MODEL_DIR / "1/model.onnx"
METADATA_PATH = MODEL_DIR / "1/export_metadata.json"
CONFIG_PATH = MODEL_DIR / "config.pbtxt"

for path in [MODEL_PATH, METADATA_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Build fixed-slot fused voting models with: \
python scripts/onnx/build_fused_string_voting_onnx.py --max-segments 15"
        )

with open(METADATA_PATH, "r") as file:
    metadata = json.load(file)

print(f"Model: {MODEL_PATH.relative_to(REPO_ROOT)}")
print(f"Metadata: {METADATA_PATH.relative_to(REPO_ROOT)}")
print(json.dumps(metadata["contract"], indent=2))

## Load ONNX Runtime Session

Because the graph contains an ONNX Runtime Extensions tokenizer custom op, we must register the ORT Extensions shared library before creating the session.


In [ ]:
ortx_library = Path(get_library_path())
if not ortx_library.exists():
    raise FileNotFoundError(f"ORT Extensions library not found: {ortx_library}")

session_options = ort.SessionOptions()
session_options.register_custom_ops_library(str(ortx_library))
session = ort.InferenceSession(str(MODEL_PATH), session_options, providers=["CPUExecutionProvider"])

print(f"Registered ORT Extensions: {ortx_library}")

In [ ]:
print("Inputs:")
for item in session.get_inputs():
    print({"name": item.name, "type": item.type, "shape": item.shape})

print("\nOutputs:")
for item in session.get_outputs():
    print({"name": item.name, "type": item.type, "shape": item.shape})

## Helper Functions

These helpers prepare fixed-slot inputs, run the fused model, and print readable outputs.


In [ ]:
OUTPUT_NAMES = [
    "logits",
    "probabilities",
    "label",
    "transcript_probabilities",
    "transcript_label_average",
    "transcript_label_majority",
]
LABELS = {0: "Healthy", 1: "CHR-P"}


def prepare_fixed_slots(segments, max_segments=15):
    if len(segments) > max_segments:
        print(f"Warning: truncating {len(segments)} segments to {max_segments}.")
    active_segments = segments[:max_segments]
    summaries = [segment["summary"] for segment in active_segments]
    text = summaries + [""] * (max_segments - len(summaries))
    segment_mask = [1.0] * len(summaries) + [0.0] * (max_segments - len(summaries))
    return np.array(text, dtype=object), np.array(segment_mask, dtype=np.float32), active_segments


def run_fused_voting(text, segment_mask):
    outputs = session.run(OUTPUT_NAMES, {"text": text, "segment_mask": segment_mask})
    return dict(zip(OUTPUT_NAMES, outputs))


def print_segment_table(active_segments, result):
    probabilities = result["probabilities"]
    labels = result["label"]
    print("slot | active | domain | label | P(Healthy) | P(CHR-P) | summary preview")
    print("-" * 110)
    for index in range(MAX_SEGMENTS):
        active = index < len(active_segments)
        domain = active_segments[index]["domain"] if active else "<padded>"
        preview = active_segments[index]["summary"][:60].replace("\n", " ") if active else ""
        label = LABELS[int(labels[index])]
        print(
            f"{index:>4} | {str(active):>6} | {domain:<24} | {label:<7} | "
            f"{probabilities[index, 0]:>10.4f} | {probabilities[index, 1]:>8.4f} | {preview}"
        )


def print_transcript_result(result):
    probs = result["transcript_probabilities"]
    avg_label = int(result["transcript_label_average"][0])
    majority_label = int(result["transcript_label_majority"][0])
    print("Transcript probabilities:", {"Healthy": float(probs[0]), "CHR-P": float(probs[1])})
    print("Average-vote label:", avg_label, LABELS[avg_label])
    print("Majority-vote label:", majority_label, LABELS[majority_label])

## Manual Segmented Example

Start with a small pre-segmented transcript. In a real application, these segment summaries would come from the CHiRPE preprocessor or another upstream segmentation/summarization step.


In [ ]:
manual_segments = [
    {
        "domain": "P1_Unusual_Thoughts",
        "summary": "The participant reports that ordinary events sometimes feel personally meaningful and connected to them.",
    },
    {
        "domain": "P2_Suspiciousness",
        "summary": "The participant describes occasional suspiciousness and worries that others may be watching them.",
    },
    {
        "domain": "P6_Experiences",
        "summary": "The participant mentions unusual perceptual experiences but reports they are intermittent.",
    },
]

text, segment_mask, active_segments = prepare_fixed_slots(manual_segments, max_segments=MAX_SEGMENTS)

print("text shape:", text.shape, text.dtype)
print("segment_mask shape:", segment_mask.shape, segment_mask.dtype)
print("active slots:", int(segment_mask.sum()))
print("first five text slots:")
for item in text[:5]:
    print(repr(item))

In [ ]:
manual_result = run_fused_voting(text, segment_mask)

print_segment_table(active_segments, manual_result)
print()
print_transcript_result(manual_result)

## Validate ONNX Voting With NumPy

The ONNX graph computes masked transcript voting. We can recompute the same logic in NumPy to confirm that padded slots do not affect the final transcript outputs.


In [ ]:
probabilities = manual_result["probabilities"]
labels = manual_result["label"]

masked_probabilities = probabilities * segment_mask[:, None]
np_transcript_probabilities = masked_probabilities.sum(axis=0) / max(float(segment_mask.sum()), 1e-6)
np_average_label = int(np_transcript_probabilities.argmax())
np_majority_label = int((labels * segment_mask.astype(np.int64)).sum() * 2 > segment_mask.sum())

checks = {
    "transcript_probabilities": np.allclose(
        manual_result["transcript_probabilities"], np_transcript_probabilities, atol=1e-6
    ),
    "average_label": int(manual_result["transcript_label_average"][0]) == np_average_label,
    "majority_label": int(manual_result["transcript_label_majority"][0]) == np_majority_label,
}

print(json.dumps(checks, indent=2))
assert all(checks.values())

## Example With A Synthetic Raw Transcript

The fixed-slot ONNX model does not perform raw transcript segmentation. We still run CHiRPE preprocessing outside ONNX, then pass the resulting segment summaries to the fused model.


In [ ]:
DATA_FILE = REPO_ROOT / "data/synthetic/test.json"
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Missing synthetic test data: {DATA_FILE}")

with open(DATA_FILE, "r") as file:
    test_items = json.load(file)

sample = test_items[0]
print({"participant_id": sample["participant_id"], "label": sample["label"], "turns": len(sample["transcript"])})

In [ ]:
preprocessor = TranscriptPreprocessor(segmentation_threshold=0.80, use_llm_summarizer=False)
processed = preprocessor.process_transcript(sample["transcript"], participant_id=sample["participant_id"])
synthetic_segments = processed["segments"]

print(f"Extracted {len(synthetic_segments)} segments.")
for index, segment in enumerate(synthetic_segments[:MAX_SEGMENTS], start=1):
    print(f"{index:>2}. {segment['domain']}: {segment['summary'][:90]}")

In [ ]:
synthetic_text, synthetic_mask, synthetic_active_segments = prepare_fixed_slots(
    synthetic_segments, max_segments=MAX_SEGMENTS
)
synthetic_result = run_fused_voting(synthetic_text, synthetic_mask)

print_segment_table(synthetic_active_segments, synthetic_result)
print()
print_transcript_result(synthetic_result)

## Triton Request Shape

The generated `config.pbtxt` uses `max_batch_size: 0`, so the model receives one transcript request at a time. A Triton HTTP request has two inputs: `text` and `segment_mask`.


In [ ]:
triton_payload = {
    "inputs": [
        {
            "name": "text",
            "shape": [MAX_SEGMENTS],
            "datatype": "BYTES",
            "data": text.tolist(),
        },
        {
            "name": "segment_mask",
            "shape": [MAX_SEGMENTS],
            "datatype": "FP32",
            "data": segment_mask.tolist(),
        },
    ],
    "outputs": [
        {"name": "transcript_probabilities"},
        {"name": "transcript_label_average"},
        {"name": "transcript_label_majority"},
    ],
}

print(json.dumps(triton_payload, indent=2)[:2500] + "\n...")

## ONNX Graphs In Plain Language

An ONNX model is a graph. A graph is a list of operations, called nodes, connected by named tensors.

A tiny graph looks like this:

```text
input tensor -> node -> output tensor
```

For this CHiRPE model, there are three conceptual graph pieces:

```text
string text -> tokenizer graph -> token tensors
token tensors -> classifier graph -> logits
logits + segment_mask -> voting graph -> transcript outputs
```

The tokenizer graph uses an ONNX Runtime Extensions custom op such as `BertTokenizer`. That is why ORT Extensions must be registered before loading the model.


## How Transcript Voting Was Merged Into The Fused Model

The fixed-slot fused voting builder wires several graph pieces together. The merge is easier to understand as a sequence of graph edits:

1. Create 15 tokenizer branches, one for each segment slot.
2. For slot `i`, slice `text[i:i+1]` from the input string tensor.
3. Run the tokenizer custom op for that one string.
4. Pad and slice each token output to the fixed sequence length, for example 128 tokens.
5. Unsqueeze each token vector into one row shaped `[1, max_length]`.
6. Concatenate the 15 rows into classifier inputs shaped `[15, max_length]`.
7. Feed those tensors into the classifier graph.
8. Add `Softmax` and `ArgMax` nodes to produce segment probabilities and labels.
9. Add `Mul`, `ReduceSum`, `Max`, and `Div` nodes to compute masked average transcript probabilities.
10. Add `ReduceSum`, `Greater`, `Cast`, and `Unsqueeze` nodes to compute masked binary majority voting.

Diagram:

```text
text[15] --+--> slice slot 0  -> tokenizer -> pad/slice -> row 0 --+
          +--> slice slot 1  -> tokenizer -> pad/slice -> row 1 --+
          +--> ...                                             +--> concat tokens -> classifier -> logits
          +--> slice slot 14 -> tokenizer -> pad/slice -> row14 --+

logits -> softmax -> probabilities -> argmax -> segment labels

probabilities + segment_mask -> masked reduce -> transcript_probabilities -> average label
segment labels + segment_mask -> masked reduce -> majority label
```

This is why the model can receive strings directly while still returning transcript-level predictions from ONNX.


## Practical Notes

- `max_segments=15` matches the 15 PSYCHS domains.
- If only 2 segments exist, pass 13 empty strings and mask them as `0.0`.
- Padded slots are ignored by transcript voting, but they still add compute cost.
- This model handles one transcript per request. It does not batch multiple transcripts in one request.
- Raw transcript segmentation and optional summarization still happen outside ONNX.
- The tokenizer is inside ONNX, so the serving client sends strings, not token IDs.


## Summary

This tutorial showed how to:

1. Load a fixed-slot fused string-input ONNX model.
2. Register ONNX Runtime Extensions for tokenizer custom ops.
3. Prepare `text[15]` and `segment_mask[15]`.
4. Run tokenizer, classifier, and transcript voting inside ONNX.
5. Validate masked voting against NumPy.
6. Understand how the tokenizer, classifier, and voting graphs were merged.
